# OpenRouter-Assisted Analysis: πολιτεία in Philo of Alexandria

This notebook uses the Perseus MCP tools to collect evidence for how Philo of Alexandria uses the Greek term `πολιτεία` and related forms, then asks an OpenRouter-hosted LLM to synthesize the evidence.

The workflow is deliberately evidence-first:

1. discover/verify the Philo textgroup;
2. search Scaife for lemma and form/operator variants scoped to Philo;
3. fetch compact passage text for selected hits;
4. build a cited evidence packet with URNs;
5. ask the LLM for an interpretation that must cite the supplied URNs and avoid claims beyond the evidence.

> Requirements: install project dependencies, have internet access to Perseus/Scaife and OpenRouter, and provide an OpenRouter API key. The notebook reads the key at runtime and does not print it. Clear outputs before committing after a credentialed run.


## Configuration

Copy `.env.example` to `.env` in the project root and set:

```dotenv
OPENROUTER_API_KEY=sk-or-v1-...
```

You can optionally set `OPENROUTER_MODEL`. The default below uses the same free model as notebook `06_`, but you can choose a stronger model for better philological synthesis.


In [ ]:
from pathlib import Path
from getpass import getpass
import html
import importlib
import json
import os
import re
import sys

import httpx
from dotenv import load_dotenv
from fastmcp import Client
from IPython.display import Markdown, display

START = Path.cwd().resolve()
REPO_ROOT = START
for candidate in [START, *START.parents]:
    if (candidate / "server.py").exists():
        REPO_ROOT = candidate
        break
else:
    raise RuntimeError(f"Could not find server.py from {START}")

sys.path.insert(0, str(REPO_ROOT))
load_dotenv(REPO_ROOT / ".env", override=False)
os.environ.setdefault("PERSEUS_MCP_CACHE_DIR", str(REPO_ROOT / ".cache" / "perseus-mcp"))

import server

server = importlib.reload(server)
mcp = server.mcp

OPENROUTER_URL = "https://openrouter.ai/api/v1/chat/completions"
OPENROUTER_MODEL = os.getenv(
    "OPENROUTER_MODEL", "nvidia/nemotron-3-super-120b-a12b:free"
)
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY") or getpass("OpenRouter API key: ")
if not OPENROUTER_API_KEY:
    raise RuntimeError("OPENROUTER_API_KEY is required.")

print(f"Repository root: {REPO_ROOT}")
print(f"Cache directory: {os.environ['PERSEUS_MCP_CACHE_DIR']}")
print(f"OpenRouter model: {OPENROUTER_MODEL}")


## Helpers

These helpers keep the notebook focused on the research workflow. `call_json` and `call_text` invoke local MCP tools. `search_rows` turns Scaife search results into compact, cited evidence rows suitable for prompting an LLM.


In [ ]:
TAG_RE = re.compile(r"<[^>]+>")


def tool_text(result):
    return "\n".join(
        block.text for block in result.content if getattr(block, "text", None) is not None
    )


async def call_json(client, tool_name, arguments=None):
    result = await client.call_tool(tool_name, arguments or {})
    return json.loads(tool_text(result))


async def call_text(client, tool_name, arguments=None):
    result = await client.call_tool(tool_name, arguments or {})
    return tool_text(result)


def clean_snippet(value):
    return html.unescape(TAG_RE.sub("", value or "")).strip()


def passage_labels(passage):
    text = passage.get("text", {})
    labels = []
    for ancestor in text.get("ancestors", []) or []:
        label = ancestor.get("label")
        if label:
            labels.append(label)
    if text.get("label"):
        labels.append(text["label"])
    return labels


def search_rows(search_data, source_label, limit=12):
    rows = []
    for result in search_data.get("results", [])[:limit]:
        passage = result.get("passage", {})
        rows.append(
            {
                "source": source_label,
                "urn": passage.get("urn"),
                "labels": passage_labels(passage),
                "snippet": clean_snippet(" ".join(result.get("content", []))),
            }
        )
    return rows


def dedupe_rows(rows):
    seen = set()
    unique = []
    for row in rows:
        urn = row.get("urn")
        if not urn or urn in seen:
            continue
        seen.add(urn)
        unique.append(row)
    return unique


def openrouter_chat(messages, temperature=0.2):
    response = httpx.post(
        OPENROUTER_URL,
        headers={
            "Authorization": f"Bearer {OPENROUTER_API_KEY}",
            "Content-Type": "application/json",
        },
        json={
            "model": OPENROUTER_MODEL,
            "messages": messages,
            "temperature": temperature,
        },
        timeout=90.0,
    )
    if response.status_code >= 400:
        raise RuntimeError(f"OpenRouter error {response.status_code}: {response.text[:2000]}")
    return response.json()["choices"][0]["message"]["content"]


## Confirm the Philo Scope

The Scaife library identifies Philo Judaeus / Philo of Alexandria with the textgroup `urn:cts:greekLit:tlg0018`. The local CTS capability feed used by some Perseus MCP discovery tools may not advertise this textgroup, so this notebook treats Scaife library metadata as the scope checkpoint and uses the textgroup URN directly for server-side search filtering.


In [ ]:
PHILO_TEXTGROUP = "urn:cts:greekLit:tlg0018"

async with Client(mcp) as client:
    philo_metadata = await call_json(
        client, "get_scaife_library_metadata", {"urn": PHILO_TEXTGROUP}
    )
    cts_name_candidates = await call_json(
        client,
        "find_author_names",
        {"query": "Philo", "language": "greek", "limit": 20},
    )

print("Scaife textgroup URN:", PHILO_TEXTGROUP)
print("Scaife label/title:", philo_metadata.get("label") or philo_metadata.get("title") or philo_metadata.get("name"))

exact_cts_matches = [
    author for author in cts_name_candidates.get("authors", [])
    if author.get("urn") == PHILO_TEXTGROUP
]
if exact_cts_matches:
    print("Local CTS discovery also contains the Philo textgroup.")
else:
    print("Local CTS author-name discovery did not return the Philo textgroup; continuing with the Scaife scope.")

print(json.dumps(philo_metadata, ensure_ascii=False, indent=2)[:2000])


## Search for πολιτεία Evidence

The search combines two approaches:

- lemma search for `πολιτεία`, which should group inflected forms under the lexical headword;
- wildcard form search for `πολιτει*`, which can catch visible surface forms and related spellings that may not be covered by lemma indexing.

Both searches are scoped server-side to Philo's textgroup.


In [ ]:
async with Client(mcp) as client:
    lemma_search = await call_json(
        client,
        "search_perseus",
        {
            "query": "πολιτεία",
            "language": "greek",
            "query_format": "unicode",
            "search_kind": "lemma",
            "text_group": PHILO_TEXTGROUP,
            "result_format": "instances",
        },
    )
    wildcard_search = await call_json(
        client,
        "search_perseus",
        {
            "query": "πολιτει*",
            "language": "greek",
            "query_format": "unicode",
            "search_kind": "form",
            "preserve_operators": True,
            "text_group": PHILO_TEXTGROUP,
            "result_format": "instances",
        },
    )

print("Lemma search total:", lemma_search.get("total_count"))
print("Wildcard form search total:", wildcard_search.get("total_count"))

evidence_rows = dedupe_rows(
    search_rows(lemma_search, "lemma: πολιτεία")
    + search_rows(wildcard_search, "form wildcard: πολιτει*")
)

assert evidence_rows, "No evidence rows found; inspect the query or upstream Scaife data"
print(f"Evidence rows selected: {len(evidence_rows)}")
for row in evidence_rows[:8]:
    print("-", row["urn"], "|", " > ".join(row["labels"]), "|", row["snippet"][:160])


## Fetch Passage Text

Search snippets are useful, but the LLM should see more context. This cell fetches Scaife plaintext for the first selected passages and builds a compact evidence packet. Increase `MAX_PASSAGES` if you want a broader sample and your chosen model has enough context window.


In [ ]:
MAX_PASSAGES = 10

async with Client(mcp) as client:
    for row in evidence_rows[:MAX_PASSAGES]:
        try:
            row["passage_text"] = await call_text(
                client, "get_scaife_passage_text", {"urn": row["urn"]}
            )
        except Exception as exc:
            row["passage_text"] = f"[Could not fetch passage text: {exc}]"

evidence_packet = {
    "research_question": "How does Philo of Alexandria use πολιτεία / politeia?",
    "scope": {
        "author_textgroup": PHILO_TEXTGROUP,
        "lemma_total_count": lemma_search.get("total_count"),
        "wildcard_total_count": wildcard_search.get("total_count"),
        "sampled_passages": min(MAX_PASSAGES, len(evidence_rows)),
    },
    "evidence": evidence_rows[:MAX_PASSAGES],
}

print(json.dumps(evidence_packet, ensure_ascii=False, indent=2)[:5000])


## Ask OpenRouter for a Cited Assessment

The LLM is instructed to use only the supplied evidence packet and to cite CTS URNs. This keeps the answer auditable: if a claim is not supported by a listed passage, it should be framed as a hypothesis or omitted.


In [ ]:
system_prompt = """You are a careful scholar of Hellenistic Jewish Greek.
Use only the evidence supplied by the user. Do not invent passages, works, or
translations. Cite CTS URNs for every substantive claim. If the evidence is
too small for a conclusion, say so explicitly. Distinguish lexical meaning,
political/institutional usage, ethical way-of-life usage, and biblical/civic
identity usage when the evidence supports those distinctions."""

user_prompt = f"""Assess how Philo of Alexandria uses πολιτεία / politeia.

Please produce:
1. a concise thesis;
2. 3-5 usage categories, each with cited URNs;
3. notes on ambiguity or limits of the sample;
4. follow-up searches that would strengthen the analysis.

Evidence packet JSON:
{json.dumps(evidence_packet, ensure_ascii=False, indent=2)}"""

analysis_markdown = openrouter_chat(
    [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]
)

display(Markdown(analysis_markdown))


## Optional: Ask for a Skeptical Review

The second LLM pass asks for a critique of the first answer against the same evidence. This is useful because the model may over-generalize from a small sample. Keep this as a separate step so you can inspect the first answer before running the critique.


In [ ]:
RUN_CRITIC_PASS = False

if RUN_CRITIC_PASS:
    critic_prompt = f"""Review the following analysis for unsupported claims.
Point out where it exceeds the evidence, misses distinctions, or needs more
passages. Cite the evidence URNs when possible.

Evidence packet:
{json.dumps(evidence_packet, ensure_ascii=False, indent=2)}

Analysis to review:
{analysis_markdown}"""
    critique = openrouter_chat(
        [
            {"role": "system", "content": "You are a skeptical philological reviewer. Use only the supplied evidence."},
            {"role": "user", "content": critic_prompt},
        ],
        temperature=0.1,
    )
    display(Markdown(critique))
else:
    print("Skipping critic pass. Set RUN_CRITIC_PASS = True to run it.")


## Suggested Follow-Up

Good follow-up searches include:

- inflected form searches for `πολιτείας`, `πολιτείᾳ`, `πολιτείαν`, and `πολιτεῖαι`;
- related vocabulary such as `νόμος`, `πόλις`, `πολίτης`, `πολιτεύομαι`, and `βίος`;
- work-scoped searches if one Philo treatise dominates the results;
- comparison with Josephus or Plato using the same evidence-packet pattern.
